# Breast Cancer Detection - Google Colab Demo
## Multi-Objective Optimization with NSGA-III

This notebook demonstrates the complete pipeline for breast cancer detection under dataset shift.

**Steps:**
1. Setup environment and install dependencies
2. Mount Google Drive and load datasets
3. Test preprocessing pipeline
4. Test model training
5. Run NSGA-III optimization (small scale)
6. Zero-shot evaluation on INbreast

## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q pydicom opencv-python-headless scikit-image
!pip install -q pymoo
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Import core libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive and Setup Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Upload the breast_cancer_detection folder to your Google Drive
# Then update this path to point to it

PROJECT_PATH = "/content/drive/MyDrive/breast_cancer_detection"

# Verify the path exists
if os.path.exists(PROJECT_PATH):
    print(f"✓ Project found at: {PROJECT_PATH}")
    # Add to Python path
    sys.path.insert(0, PROJECT_PATH)
else:
    print(f"✗ Project not found at: {PROJECT_PATH}")
    print("Please upload the breast_cancer_detection folder to your Google Drive")

In [ ]:
# Configure data paths
# UPDATE THESE PATHS to match your Google Drive structure

VINDR_IMAGES_ROOT = "/content/drive/MyDrive/vindr-mammo/images"
VINDR_CSV = "/content/drive/MyDrive/vindr-mammo/metadata/stratified_selection.csv"

INBREAST_DICOM_DIR = "/content/drive/MyDrive/INbreast/AllDICOMs"
INBREAST_CSV = "/content/drive/MyDrive/INbreast/INbreast.csv"

# Verify paths
print("Checking data paths...")
print(f"VinDr images: {os.path.exists(VINDR_IMAGES_ROOT)}")
print(f"VinDr CSV: {os.path.exists(VINDR_CSV)}")
print(f"INbreast DICOM: {os.path.exists(INBREAST_DICOM_DIR)}")
print(f"INbreast CSV: {os.path.exists(INBREAST_CSV)}")

## 3. Test Preprocessing Pipeline

In [ ]:
from src.preprocessing import MammographyPreprocessor

# Create preprocessor
preprocessor = MammographyPreprocessor()
print("✓ Preprocessor created successfully")

In [ ]:
# Test on a sample DICOM file
# Replace with an actual path from your dataset
sample_dicom = "/content/drive/MyDrive/vindr-mammo/images/STUDY_ID/IMAGE_ID.dicom"

if os.path.exists(sample_dicom):
    processed_img = preprocessor(sample_dicom)
    
    print(f"Processed image shape: {processed_img.shape}")
    print(f"Expected shape: (480, 720, 3)")
    print(f"Value range: [{processed_img.min()}, {processed_img.max()}]")
    
    # Visualize
    plt.figure(figsize=(8, 6))
    plt.imshow(processed_img)
    plt.title("Preprocessed Mammogram")
    plt.axis('off')
    plt.show()
else:
    print(f"Sample file not found. Please update the path.")

## 4. Test Dataset Loading

In [ ]:
from src.datasets import VinDRMammoBinaryDataset
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split

# Load dataset
print("Loading VinDr-Mammo dataset...")
dataset = VinDRMammoBinaryDataset(
    images_root=VINDR_IMAGES_ROOT,
    csv_file=VINDR_CSV,
    preprocessor=preprocessor
)

print(f"\n✓ Dataset loaded: {len(dataset)} samples")

In [ ]:
# Create train/val split
labels = [dataset.samples[i][-1] for i in range(len(dataset))]
indices = np.arange(len(dataset))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

# Compute class distribution
train_labels = [labels[i] for i in train_idx]
n_benign = sum(1 for l in train_labels if l == 0)
n_malignant = sum(1 for l in train_labels if l == 1)

print(f"\nTrain: {len(train_dataset)} samples")
print(f"  Benign: {n_benign}")
print(f"  Malignant: {n_malignant}")
print(f"  Positive weight: {n_benign/n_malignant:.3f}")

print(f"\nValidation: {len(val_dataset)} samples")

In [ ]:
# Test loading a sample
img, label = dataset[0]

print(f"\nSample data:")
print(f"  Image shape: {img.shape}")
print(f"  Image dtype: {img.dtype}")
print(f"  Value range: [{img.min():.3f}, {img.max():.3f}]")
print(f"  Label: {label.item()} ({'Malignant' if label.item() == 1 else 'Benign'})")

# Visualize
plt.figure(figsize=(8, 6))
plt.imshow(img.permute(1, 2, 0))
plt.title(f"Sample Image - {'Malignant' if label.item() == 1 else 'Benign'}")
plt.axis('off')
plt.show()

## 5. Test Model Building

In [ ]:
from src.models import build_resnet152

# Build model with different configurations
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Testing model configurations...\n")

# Test 1: Full fine-tuning
model1 = build_resnet152(
    pretrained=True,
    dropout=0.2,
    unfreeze_fraction=1.0
)
info1 = model1.get_trainable_params_info()
print("Config 1 - Full fine-tuning (unfreeze=1.0):")
print(f"  Total params: {info1['total_params']:,}")
print(f"  Trainable: {info1['trainable_params']:,} ({info1['trainable_percentage']:.1f}%)\n")

# Test 2: Partial fine-tuning
model2 = build_resnet152(
    pretrained=True,
    dropout=0.2,
    unfreeze_fraction=0.5
)
info2 = model2.get_trainable_params_info()
print("Config 2 - Partial fine-tuning (unfreeze=0.5):")
print(f"  Total params: {info2['total_params']:,}")
print(f"  Trainable: {info2['trainable_params']:,} ({info2['trainable_percentage']:.1f}%)\n")

# Test 3: Feature extraction only
model3 = build_resnet152(
    pretrained=True,
    dropout=0.2,
    unfreeze_fraction=0.0
)
info3 = model3.get_trainable_params_info()
print("Config 3 - Feature extraction (unfreeze=0.0):")
print(f"  Total params: {info3['total_params']:,}")
print(f"  Trainable: {info3['trainable_params']:,} ({info3['trainable_percentage']:.1f}%)")

# Test forward pass
model1 = model1.to(device)
test_input = torch.randn(2, 3, 480, 720).to(device)
output = model1(test_input)
print(f"\n✓ Forward pass successful: Input {test_input.shape} → Output {output.shape}")

## 6. Test Augmentation

In [ ]:
from src.augmentations import get_augmentation

# Test different augmentation strengths
img_sample = dataset[0][0]  # Get first image

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

strengths = [0.0, 0.3, 0.6, 1.0]

for i, strength in enumerate(strengths):
    aug = get_augmentation(strength)
    img_aug = aug(img_sample.clone())
    
    axes[i].imshow(img_aug.permute(1, 2, 0))
    axes[i].set_title(f"Augmentation Strength: {strength}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("✓ Augmentation test completed")

## 7. Test Training Pipeline (Quick Demo)

In [ ]:
from src.training import train_model
from torch.utils.data import DataLoader

# Create small subset for quick testing (10% of data)
small_train_idx = train_idx[:len(train_idx)//10]
small_val_idx = val_idx[:len(val_idx)//10]

small_train_dataset = Subset(dataset, small_train_idx)
small_val_dataset = Subset(dataset, small_val_idx)

print(f"Quick test with {len(small_train_dataset)} train, {len(small_val_dataset)} val samples")

# Create dataloaders
train_loader = DataLoader(
    small_train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    small_val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Build model
model = build_resnet152(
    pretrained=True,
    dropout=0.2,
    unfreeze_fraction=0.5  # Partial fine-tuning for speed
).to(device)

print("\nStarting quick training test (5 epochs max)...\n")

In [ ]:
# Train for a few epochs
model, metrics = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    val_dataset=small_val_dataset,
    device=device,
    learning_rate=1e-4,
    weight_decay=1e-4,
    patience=3,
    max_epochs=5,  # Just 5 epochs for testing
    pos_weight=n_benign/n_malignant,
    verbose=True
)

print("\n" + "="*80)
print("Training test completed!")
print(f"Final PR-AUC: {metrics['pr_auc']:.4f}")
print(f"Final AUROC: {metrics['auroc']:.4f}")
print(f"Final Brier: {metrics['brier']:.4f}")
print(f"Robustness Degradation: {metrics['robustness_degradation']:.4f}")
print("="*80)

## 8. Test Breast-Level Aggregation (Noisy-OR)

In [ ]:
from src.evaluation import noisy_or_aggregation, aggregate_breast_level_predictions

# Test Noisy-OR formula
print("Testing Noisy-OR aggregation formula:\n")

# Example: CC and MLO views of same breast
test_cases = [
    ([0.2, 0.3], "Low probabilities"),
    ([0.7, 0.8], "High probabilities"),
    ([0.1, 0.9], "Mixed probabilities"),
    ([0.5, 0.5], "Equal probabilities")
]

for probs, description in test_cases:
    result = noisy_or_aggregation(probs)
    print(f"{description}:")
    print(f"  Views: CC={probs[0]:.2f}, MLO={probs[1]:.2f}")
    print(f"  Breast-level: {result:.4f}")
    print(f"  Formula check: 1 - (1-{probs[0]:.2f})*(1-{probs[1]:.2f}) = {result:.4f}\n")

In [ ]:
# Test on real dataset
print("\nTesting breast-level aggregation on validation set...\n")

y_true_breast, y_probs_breast = aggregate_breast_level_predictions(
    model, small_val_dataset, device
)

print(f"Number of breasts evaluated: {len(y_true_breast)}")
print(f"Benign breasts: {sum(y_true_breast == 0)}")
print(f"Malignant breasts: {sum(y_true_breast == 1)}")
print(f"\nBreast-level probability range: [{y_probs_breast.min():.3f}, {y_probs_breast.max():.3f}]")

## 9. Test Robustness Evaluation

In [ ]:
from src.robustness import RobustnessTester

# Test robustness degradation
print("Testing robustness to perturbations...\n")

tester = RobustnessTester(
    brightness_delta=0.1,
    contrast_factor=0.1,
    noise_std=0.02
)

results = tester.evaluate_robustness(model, val_loader, device)

print(f"PR-AUC (clean): {results['pr_auc_standard']:.4f}")
print(f"PR-AUC (perturbed): {results['pr_auc_perturbed']:.4f}")
print(f"Robustness Degradation: {results['degradation']:.4f}")
print(f"\nLower degradation = more robust model")

## 10. NSGA-III Optimization (Small Scale Demo)

**Warning:** Full optimization with 20 population × 50 generations = 1000 evaluations will take 20-50 hours.

This demo runs a **small-scale** version (5 population × 3 generations = 15 evaluations) for demonstration.

In [ ]:
from src.optimization import BreastCancerOptimizationProblem
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.optimize import minimize
from pymoo.util.ref_dirs import get_reference_directions

print("Setting up NSGA-III optimization (DEMO VERSION - small scale)...\n")

# Create problem
problem = BreastCancerOptimizationProblem(
    train_dataset=small_train_dataset,  # Using small subset
    val_dataset=small_val_dataset,
    device=device,
    batch_size=4,
    num_workers=2,
    patience=3,
    max_epochs=5,  # Reduced epochs for demo
    pos_weight=n_benign/n_malignant,
    random_seed=42
)

# Generate reference directions for 4 objectives
ref_dirs = get_reference_directions("das-dennis", 4, n_partitions=3)
print(f"Reference directions: {len(ref_dirs)}")

# Create algorithm
algorithm = NSGA3(
    ref_dirs=ref_dirs,
    pop_size=5  # Small population for demo
)

print("\nStarting optimization...")
print("This will take ~15-30 minutes for 5 pop × 3 gen = 15 evaluations\n")

In [ ]:
# Run optimization
res = minimize(
    problem,
    algorithm,
    termination=("n_gen", 3),  # Just 3 generations for demo
    seed=42,
    verbose=True
)

print("\n" + "="*80)
print("Optimization completed!")
print(f"Pareto solutions found: {len(res.F)}")
print("="*80)

In [ ]:
# Display Pareto front
print("\nPareto Front Solutions:\n")
print(f"{'ID':<5} {'PR-AUC':>8} {'AUROC':>8} {'Brier':>8} {'Robust':>8}")
print("-" * 45)

for i, f in enumerate(res.F):
    pr_auc = -f[0]  # Convert back from minimization
    auroc = -f[1]
    brier = f[2]
    robust = f[3]
    
    print(f"{i:<5} {pr_auc:>8.4f} {auroc:>8.4f} {brier:>8.4f} {robust:>8.4f}")

print("\nNote: This is a DEMO with reduced scale.")
print("For production, use: pop_size=20, n_gen=50 in run_nsga3.py")

## 11. Visualize Pareto Front

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Extract objectives
pr_auc = -res.F[:, 0]
auroc = -res.F[:, 1]
brier = res.F[:, 2]
robust = res.F[:, 3]

# 2D scatter plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# PR-AUC vs AUROC
axes[0, 0].scatter(pr_auc, auroc, c='blue', s=100)
axes[0, 0].set_xlabel('PR-AUC')
axes[0, 0].set_ylabel('AUROC')
axes[0, 0].set_title('PR-AUC vs AUROC')
axes[0, 0].grid(True)

# PR-AUC vs Brier
axes[0, 1].scatter(pr_auc, brier, c='red', s=100)
axes[0, 1].set_xlabel('PR-AUC')
axes[0, 1].set_ylabel('Brier Score')
axes[0, 1].set_title('PR-AUC vs Brier')
axes[0, 1].grid(True)

# PR-AUC vs Robustness
axes[0, 2].scatter(pr_auc, robust, c='green', s=100)
axes[0, 2].set_xlabel('PR-AUC')
axes[0, 2].set_ylabel('Robustness Degradation')
axes[0, 2].set_title('PR-AUC vs Robustness')
axes[0, 2].grid(True)

# AUROC vs Brier
axes[1, 0].scatter(auroc, brier, c='purple', s=100)
axes[1, 0].set_xlabel('AUROC')
axes[1, 0].set_ylabel('Brier Score')
axes[1, 0].set_title('AUROC vs Brier')
axes[1, 0].grid(True)

# AUROC vs Robustness
axes[1, 1].scatter(auroc, robust, c='orange', s=100)
axes[1, 1].set_xlabel('AUROC')
axes[1, 1].set_ylabel('Robustness Degradation')
axes[1, 1].set_title('AUROC vs Robustness')
axes[1, 1].grid(True)

# Brier vs Robustness
axes[1, 2].scatter(brier, robust, c='brown', s=100)
axes[1, 2].set_xlabel('Brier Score')
axes[1, 2].set_ylabel('Robustness Degradation')
axes[1, 2].set_title('Brier vs Robustness')
axes[1, 2].grid(True)

plt.tight_layout()
plt.show()

## 12. Test INbreast Dataset Loading

In [ ]:
from src.datasets import INbreastDataset

# Load INbreast dataset
print("Loading INbreast dataset...\n")

inbreast_dataset = INbreastDataset(
    dicom_dir=INBREAST_DICOM_DIR,
    csv_file=INBREAST_CSV,
    preprocessor=preprocessor
)

print(f"✓ INbreast dataset loaded: {len(inbreast_dataset)} samples")

# Test sample
img, label = inbreast_dataset[0]
print(f"\nSample shape: {img.shape}")
print(f"Sample label: {label.item()}")

# Visualize
plt.figure(figsize=(8, 6))
plt.imshow(img.permute(1, 2, 0))
plt.title(f"INbreast Sample - {'Malignant' if label.item() == 1 else 'Benign'}")
plt.axis('off')
plt.show()

## 13. Summary & Next Steps

### ✓ All Components Tested Successfully

1. **Preprocessing Pipeline** - Converts DICOM to 720×480 RGB with magma colormap
2. **Dataset Loading** - VinDr-Mammo and INbreast datasets working
3. **Model Architecture** - ResNet152 with partial fine-tuning control
4. **Augmentation** - Intensity-based augmentation with strength controller
5. **Training Pipeline** - Early stopping, class balancing, breast-level evaluation
6. **Robustness Testing** - Perturbation-based robustness measurement
7. **NSGA-III Optimization** - Multi-objective hyperparameter optimization

### 🚀 To Run Full-Scale Optimization:

```bash
cd /content/drive/MyDrive/breast_cancer_detection

# Run full NSGA-III (20-50 hours)
python scripts/run_nsga3.py \
    --pop_size 20 \
    --n_gen 50 \
    --max_epochs 100 \
    --run_id "full_run_001"

# Evaluate on INbreast (requires saved model checkpoints)
python scripts/evaluate_zeroshot.py \
    --results_file checkpoints/nsga3_results_full_run_001.pkl \
    --checkpoint_dir checkpoints/ \
    --run_id "full_run_001"
```

### 📊 Expected Runtime:
- Single model training: 10-60 minutes (depends on early stopping)
- Full NSGA-III (1000 evaluations): 20-50 hours on GPU
- Zero-shot evaluation: 1-2 hours for all Pareto solutions

### 📝 Results Will Be Saved In:
- `logs/nsga3_run_*.csv` - Hyperparameters and objectives per evaluation
- `checkpoints/nsga3_results_*.pkl` - Pareto front solutions
- `logs/zeroshot_evaluation_*.csv` - INbreast evaluation results